# Dino Segmentation Model (YOLOv8-seg)

Trains a pixel-perfect segmentation model to detect the toy dino in any
camera frame. Replaces the LiDAR+ICP detection in the iPad app.

**Pipeline:**
1. Download Josh's dino photos from Dropbox
2. Use **GroundedSAM** (GroundingDINO + SAM) with text prompt "toy dinosaur"
   to automatically generate a pixel-perfect mask in each photo
3. Convert masks to YOLO segmentation format
4. Fine-tune YOLOv8-seg on the dataset
5. Export to Core ML for iPad Neural Engine

**Runtime:** A100 GPU. Total time ~45-60 min including auto-masking.

## Cell 1: Setup

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Install deps
!pip install -q ultralytics coremltools Pillow opencv-python-headless tqdm matplotlib
# GroundedSAM — text-prompted segmentation via GroundingDINO + SAM
!pip install -q autodistill autodistill-grounded-sam supervision

import os
os.makedirs('/content/dino_project', exist_ok=True)
os.chdir('/content/dino_project')
print(f'Working directory: {os.getcwd()}')

## Cell 2: Download Dino Photos

In [ ]:
import os

photos_dir = '/content/dino_project/photos'
os.makedirs(photos_dir, exist_ok=True)

zip_path = '/content/dino_project/dino_photos.zip'
if not os.path.exists(zip_path):
    print('Downloading dino photos from Dropbox...')
    !wget -q -O {zip_path} "https://www.dropbox.com/scl/fi/tj313v2izxd43wfmxq68r/dino_photos.zip?rlkey=rnr9asoothhe8e97cma6jx7r4&st=xricbskx&dl=1"
    print('Downloaded!')

print('Unzipping...')
!unzip -q -o {zip_path} -d {photos_dir}

# Handle nested folder
import glob
for subdir in glob.glob(f'{photos_dir}/*'):
    if os.path.isdir(subdir):
        !mv {subdir}/* {photos_dir}/ 2>/dev/null || true
        !rmdir {subdir} 2>/dev/null || true

# Normalize extensions to lowercase (Linux is case-sensitive — .JPG != .jpg)
renamed = 0
for f in list(os.listdir(photos_dir)):
    stem, ext = os.path.splitext(f)
    if ext != ext.lower():
        src = os.path.join(photos_dir, f)
        dst = os.path.join(photos_dir, stem + ext.lower())
        os.rename(src, dst)
        renamed += 1
if renamed > 0:
    print(f'Normalized {renamed} filenames to lowercase extensions')

# Count
num = len([f for f in os.listdir(photos_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.heic'))])
print(f'Dino photos: {num}')

# Convert HEIC to JPG if needed
heic_count = len([f for f in os.listdir(photos_dir) if f.lower().endswith('.heic')])
if heic_count > 0:
    print(f'Converting {heic_count} HEIC images to JPG...')
    !apt-get install -qq libheif-examples 2>&1 | tail -3
    !pip install -q pillow-heif
    from pillow_heif import register_heif_opener
    register_heif_opener()
    from PIL import Image
    for f in os.listdir(photos_dir):
        if f.lower().endswith('.heic'):
            img = Image.open(os.path.join(photos_dir, f)).convert('RGB')
            jpg_name = os.path.splitext(f)[0] + '.jpg'
            img.save(os.path.join(photos_dir, jpg_name), 'JPEG', quality=95)
            os.remove(os.path.join(photos_dir, f))
    print('HEIC conversion complete')

## Cell 3: Load GroundedSAM

GroundedSAM combines **GroundingDINO** (text → bounding box) with **SAM**
(bounding box → pixel mask). Give it a text prompt like "toy dinosaur" and
it finds the object in each photo and produces a precise mask.

First run: downloads the GroundingDINO + SAM weights (~3GB total).

In [ ]:
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology

# Ontology: text prompt → class name
# Keep it simple — just one class, "dino"
ontology = CaptionOntology({
    "toy dinosaur": "dino",
    # Alternative prompts help if first one misses:
    # "plastic dinosaur": "dino",
    # "green dinosaur": "dino",
})

print('Loading GroundedSAM (downloads weights on first run)...')
base_model = GroundedSAM(ontology=ontology)
print('Ready!')

## Cell 4: Auto-Generate Dino Masks with Text Prompting

GroundedSAM processes each photo: "toy dinosaur" → bounding box → SAM mask.
Runs the entire folder in one shot. No manual clicks, no heuristics to tune.

Expected: 5-15 minutes for 296 photos on A100.

In [ ]:
import os, numpy as np, cv2
from PIL import Image
from tqdm import tqdm
import supervision as sv

masks_dir = '/content/dino_project/masks'
os.makedirs(masks_dir, exist_ok=True)

photos = sorted([f for f in os.listdir(photos_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'Processing {len(photos)} photos with GroundedSAM...')

success = 0
failed = []
for photo_name in tqdm(photos):
    photo_path = os.path.join(photos_dir, photo_name)
    mask_path = os.path.join(masks_dir, os.path.splitext(photo_name)[0] + '.png')
    if os.path.exists(mask_path):
        success += 1
        continue

    try:
        # GroundedSAM returns supervision.Detections with bounding boxes AND masks
        detections = base_model.predict(photo_path)

        if len(detections) == 0 or detections.mask is None:
            failed.append(photo_name)
            continue

        # Pick the detection with highest confidence (best dino match)
        best_idx = int(np.argmax(detections.confidence)) if detections.confidence is not None else 0
        mask = detections.mask[best_idx]  # boolean array, shape (H, W)

        # Save as PNG (255 = dino, 0 = background)
        Image.fromarray((mask.astype(np.uint8) * 255)).save(mask_path)
        success += 1
    except Exception as e:
        failed.append(photo_name)
        if len(failed) <= 3:  # log first few failures
            print(f'Error on {photo_name}: {e}')

print(f'\nGenerated masks for {success}/{len(photos)} photos')
if failed:
    print(f'Failed: {len(failed)} photos (will skip in training)')
    if len(failed) > 20:
        print(f'More than 10% failed — consider alternative prompts in Cell 3')

## Cell 5: Visual Spot-Check

Display a few random photos with their SAM-generated masks overlaid.
Verify the mask actually covers the dino before we train on thousands of iterations.

**If masks look bad:** We can tweak SAM parameters or add manual clicks for bad ones.

In [ ]:
import matplotlib.pyplot as plt, random
from PIL import Image

mask_files = sorted(os.listdir(masks_dir))
sample = random.sample(mask_files, min(6, len(mask_files)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, mask_file in enumerate(sample):
    ax = axes[idx // 3, idx % 3]
    photo_name = os.path.splitext(mask_file)[0]
    # Try common extensions
    photo_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        p = os.path.join(photos_dir, photo_name + ext)
        if os.path.exists(p):
            photo_path = p
            break
    if not photo_path: continue

    img = np.array(Image.open(photo_path).convert('RGB'))
    mask = np.array(Image.open(os.path.join(masks_dir, mask_file))) > 127

    overlay = img.copy()
    overlay[mask] = overlay[mask] * 0.5 + np.array([255, 0, 0]) * 0.5

    ax.imshow(overlay.astype(np.uint8))
    ax.set_title(photo_name)
    ax.axis('off')

plt.suptitle('SAM auto-masked dino photos (red = detected dino)', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 6: Convert Masks to YOLO Segmentation Format

YOLOv8-seg wants mask polygons in a specific text format:
```
class_id x1 y1 x2 y2 x3 y3 ...
```
Normalized coordinates [0, 1]. We convert each PNG mask to contour polygons.

In [ ]:
import cv2, shutil, random

# Set up YOLO dataset structure
dataset_dir = '/content/dino_project/dataset'
for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{dataset_dir}/labels/{split}', exist_ok=True)

mask_files = sorted(os.listdir(masks_dir))
random.seed(42)
random.shuffle(mask_files)
split_idx = int(len(mask_files) * 0.85)
train_files = mask_files[:split_idx]
val_files = mask_files[split_idx:]

def mask_to_yolo_polygons(mask_path, img_w, img_h):
    mask = np.array(Image.open(mask_path)) > 127
    mask_u8 = mask.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for c in contours:
        if len(c) < 3: continue
        # Simplify polygon to keep file sizes reasonable
        epsilon = 0.002 * cv2.arcLength(c, True)
        c = cv2.approxPolyDP(c, epsilon, True)
        if len(c) < 3: continue
        pts = c.reshape(-1, 2).astype(np.float32)
        pts[:, 0] /= img_w
        pts[:, 1] /= img_h
        polygons.append(pts.flatten().tolist())
    return polygons

def process_split(files, split):
    count = 0
    for mf in files:
        stem = os.path.splitext(mf)[0]
        photo_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            p = os.path.join(photos_dir, stem + ext)
            if os.path.exists(p):
                photo_path = p
                break
        if not photo_path: continue

        img = Image.open(photo_path)
        w, h = img.size
        polys = mask_to_yolo_polygons(os.path.join(masks_dir, mf), w, h)
        if not polys: continue

        # Copy image
        shutil.copy(photo_path, f'{dataset_dir}/images/{split}/{stem}.jpg')
        # Write label
        with open(f'{dataset_dir}/labels/{split}/{stem}.txt', 'w') as f:
            for poly in polys:
                coords = ' '.join(f'{v:.6f}' for v in poly)
                f.write(f'0 {coords}\n')
        count += 1
    return count

train_n = process_split(train_files, 'train')
val_n = process_split(val_files, 'val')
print(f'Train: {train_n}, Val: {val_n}')

# Write data.yaml
yaml_content = f'''path: {dataset_dir}
train: images/train
val: images/val
names:
  0: dino
'''
with open(f'{dataset_dir}/data.yaml', 'w') as f:
    f.write(yaml_content)
print(f'Wrote {dataset_dir}/data.yaml')

## Cell 7: Train YOLOv8-seg

In [ ]:
from ultralytics import YOLO

# Start from pretrained YOLOv8n-seg (smallest/fastest, good for mobile)
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data=f'{dataset_dir}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/dino_project/runs',
    name='dino_seg',
    patience=20,  # early stop if no improvement
    save=True,
    pretrained=True,
)
print('Training complete!')

## Cell 8: Validate Visually

In [ ]:
best_path = '/content/dino_project/runs/dino_seg/weights/best.pt'
model = YOLO(best_path)

# Run on a few val images
val_images = sorted(os.listdir(f'{dataset_dir}/images/val'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, img_name in enumerate(val_images):
    ax = axes[idx // 3, idx % 3]
    img_path = f'{dataset_dir}/images/val/{img_name}'
    res = model.predict(img_path, conf=0.25, verbose=False)
    annotated = res[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_name)
    ax.axis('off')
plt.suptitle('Trained model predictions on validation set', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 9: Export to Core ML

In [ ]:
# Export YOLOv8-seg to Core ML (nms=False for raw outputs, fp16 for Neural Engine)
model = YOLO(best_path)
coreml_path = model.export(
    format='coreml',
    imgsz=640,
    nms=True,
    half=True,
)
print(f'Exported to: {coreml_path}')

## Cell 10: Download Model

In [ ]:
from google.colab import files
import shutil

# mlpackage is a directory; zip it first
mlpackage_dir = coreml_path
zip_path = '/content/dino_project/dino_seg.mlpackage.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', mlpackage_dir)
print(f'Zipped: {zip_path}')

# Also copy the PT weights in case we want to retrain
shutil.copy(best_path, '/content/dino_project/dino_seg.pt')

print('\nDownloading...')
files.download(zip_path)
files.download('/content/dino_project/dino_seg.pt')